In [ ]:
# ============================================================
# BINARY SHIFT-AND-ADD MULTIPLIER
# ============================================================
#
# This notebook demonstrates binary multiplication using the
# classical shift-and-add algorithm.
#
# HOW TO USE THE NOTEBOOK
#
# 1. Use the "Multiplicand A" slider to select the first unsigned
#    integer.
#
# 2. Use the "Multiplier B" slider to select the second unsigned
#    integer.
#
# 3. Use the "Word length" slider to select the number of bits
#    used for representing A and B.
#
# 4. Use the "Clock / step" slider to follow the multiplication
#    process one multiplier bit at a time.
#
# 5. The multiplier is scanned from its least significant bit
#    (LSB) toward its most significant bit (MSB).
#
# 6. At each step:
#
#       - if the current multiplier bit is 0, the partial product
#         is zero,
#
#       - if the current multiplier bit is 1, the partial product
#         is the multiplicand shifted left by the current bit
#         position.
#
# 7. The current partial product is added to the cumulative sum.
#
# 8. After all multiplier bits have been processed, the complete
#    product is displayed in binary and decimal form.
#
# 9. The product of two N-bit unsigned integers may require up to
#    2N bits.
#
# 10. Whenever A, B, or the word length changes, a new
#     multiplication begins and the Clock / step slider is
#     automatically reset to 1.
#
# IMPORTANT
#
# This notebook uses unsigned binary integers so that the basic
# operation of the shift-and-add multiplier can be studied without
# the additional details required for signed-number representations.
#
# ============================================================


from ipywidgets import IntSlider, HBox, VBox, Layout, HTML
from IPython.display import display


# ------------------------------------------------------------
# Global style sheet
# ------------------------------------------------------------

style_html = HTML("""
<style>

.mult-root {
    font-family: monospace;
    width: 100%;
    max-width: 900px;
    box-sizing: border-box;
}

.mult-title {
    font-size: 22px;
    font-weight: bold;
    margin-bottom: 8px;
    color: #1f1f1f;
}

.description-box {
    font-size: 13px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 9px;
    box-sizing: border-box;
    white-space: normal;
}

.section-box {
    border: 1px solid #c8d0dc;
    border-radius: 10px;
    padding: 10px 12px;
    background: #ffffff;
    box-sizing: border-box;
    width: 100%;
}

.section-title {
    font-size: 16px;
    font-weight: bold;
    margin-bottom: 7px;
    color: #243447;
}

.info {
    font-size: 14px;
    line-height: 1.6;
}

.metric {
    display: inline-block;
    min-width: 180px;
    font-weight: bold;
    color: #243447;
}

.bit-line {
    white-space: normal;
    line-height: 34px;
    margin-top: 5px;
    margin-bottom: 5px;
}

.bit-label {
    display: inline-block;
    width: 22px;
    font-weight: bold;
}

.bit-box {
    display: inline-block;
    width: 26px;
    height: 29px;
    line-height: 29px;
    text-align: center;
    margin-right: 3px;
    border-radius: 5px;
    border: 1px solid #7f8c9a;
    font-size: 13px;
    font-weight: bold;
    box-sizing: border-box;
}

.bit-a {
    background: #dceeff;
    color: #0e3a66;
}

.bit-b {
    background: #ffe7cf;
    color: #7a3d00;
}

.bit-active {
    background: #fff3bf !important;
    color: #111111 !important;
    border: 3px solid #d62828 !important;
}

.bit-zero {
    background: #f0f2f4;
    color: #68717c;
}

.bit-product {
    background: #e8e0f3;
    color: #4b2768;
}

.bit-sum {
    background: #dff3e4;
    color: #1d5d2d;
}

.small-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    white-space: normal;
    margin-top: 5px;
}

.controls-title {
    font-family: monospace;
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.operation {
    font-size: 16px;
    font-weight: bold;
    margin-top: 6px;
    margin-bottom: 6px;
}

.result-value {
    font-size: 18px;
    font-weight: bold;
    color: #1f3b4d;
}

.step-table {
    border-collapse: collapse;
    font-family: monospace;
    font-size: 12px;
    width: 100%;
    margin-top: 6px;
    line-height: 1.2;
}

.step-table th,
.step-table td {
    border: 1px solid #c8d0dc;
    padding: 4px 6px;
    text-align: center;
    vertical-align: middle;
}

.step-table th {
    background: #f2f4f7;
}

.current-row {
    background: #fff7d6;
    font-weight: bold;
}

</style>
""")


# ------------------------------------------------------------
# Arithmetic
# ------------------------------------------------------------

def multiplication_steps(A, B, bits):
    A_binary = format(A, f'0{bits}b')
    B_binary = format(B, f'0{bits}b')

    product_bits = 2 * bits
    cumulative_sum = 0
    steps = []

    for position in range(bits):
        multiplier_bit = (B >> position) & 1

        if multiplier_bit == 1:
            partial_product = A << position
        else:
            partial_product = 0

        cumulative_sum += partial_product

        steps.append({
            'position': position,
            'multiplier_bit': multiplier_bit,
            'shift': position,
            'partial_product': partial_product,
            'cumulative_sum': cumulative_sum,
            'partial_binary': format(partial_product, f'0{product_bits}b'),
            'sum_binary': format(cumulative_sum, f'0{product_bits}b')
        })

    final_product = A * B
    final_binary = format(final_product, f'0{product_bits}b')

    return A_binary, B_binary, steps, final_product, final_binary


# ------------------------------------------------------------
# HTML helpers
# ------------------------------------------------------------

def render_bit_row(label, binary_string, css_class, active_index=None):
    boxes = []

    for i, bit in enumerate(binary_string):
        css = css_class

        if active_index is not None and i == active_index:
            css += " bit-active"

        boxes.append(f"<span class='bit-box {css}'>{bit}</span>")

    return f"""
    <div class="bit-line">
        <span class="bit-label">{label}</span>
        {''.join(boxes)}
    </div>
    """


def render_product_bits(binary_string, css_class):
    boxes = []

    for bit in binary_string:
        boxes.append(f"<span class='bit-box {css_class}'>{bit}</span>")

    return ''.join(boxes)


def build_step_table(steps, current_step):
    rows = []

    for i, item in enumerate(steps, start=1):
        css = "current-row" if i == current_step else ""

        rows.append(f"""
        <tr class="{css}">
            <td>{i}</td>
            <td>{item['position']}</td>
            <td>{item['multiplier_bit']}</td>
            <td>{item['shift']}</td>
            <td>{item['partial_product']}</td>
            <td>{item['cumulative_sum']}</td>
        </tr>
        """)

    return ''.join(rows)


# ------------------------------------------------------------
# HTML containers
# ------------------------------------------------------------

title_html = HTML("""
<div class="mult-root">
    <div class="mult-title">
        Binary Shift-and-Add Multiplier
    </div>
</div>
""")


description_html = HTML("""
<div class="mult-root">

    <div class="description-box">

        Binary multiplication is performed by scanning the multiplier
        from the <b>LSB toward the MSB</b>. If the current multiplier bit
        is 1, a shifted copy of the multiplicand is added to the
        cumulative sum; if the bit is 0, the partial product is zero.<br>

        Use the <b>Clock / step</b> slider to follow the generation,
        shifting, and accumulation of the partial products.<br>

        Changing either number or the word length automatically starts
        a new multiplication from step 1.

    </div>

</div>
""")


input_html = HTML()
step_html = HTML()
result_html = HTML()


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(width='285px')

style_opts = {'description_width': '110px'}


word_length_slider = IntSlider(
    min=3,
    max=8,
    step=1,
    value=5,
    description='Word length:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


initial_maximum = 2**word_length_slider.value - 1


a_slider = IntSlider(
    min=0,
    max=initial_maximum,
    step=1,
    value=13,
    description='Multiplicand A:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


b_slider = IntSlider(
    min=0,
    max=initial_maximum,
    step=1,
    value=11,
    description='Multiplier B:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


step_slider = IntSlider(
    min=1,
    max=word_length_slider.value,
    step=1,
    value=1,
    description='Clock / step:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


controls_title = HTML("""
<div class="controls-title">
Controls
</div>
""")


controls_box = VBox(
    [
        controls_title,
        a_slider,
        b_slider,
        word_length_slider,
        step_slider
    ],
    layout=Layout(
        width='315px',
        min_width='315px',
        border='1px solid #c8d0dc',
        padding='10px',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Dynamic display
# ------------------------------------------------------------

def update_display(*args):
    A = a_slider.value
    B = b_slider.value
    bits = word_length_slider.value
    step = step_slider.value

    A_binary, B_binary, steps, final_product, final_binary = multiplication_steps(A, B, bits)

    current = steps[step - 1]

    multiplier_active_index = bits - step


    # --------------------------------------------------------
    # Input section
    # --------------------------------------------------------

    input_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            Input Numbers
        </div>

        <div class="info">
            <span class="metric">Multiplicand</span>
            A = {A}<br>

            <span class="metric">Multiplier</span>
            B = {B}<br>

            <span class="metric">Input word length</span>
            N = {bits} bits<br>

            <span class="metric">Maximum product length</span>
            2N = {2 * bits} bits
        </div>

        <div style="margin-top:7px;">
            {render_bit_row('A', A_binary, 'bit-a')}
            {render_bit_row('B', B_binary, 'bit-b', multiplier_active_index)}
        </div>

        <div class="small-note">
            The highlighted bit of B is the multiplier bit processed
            during the current clock step.
        </div>

    </div>
    """


    # --------------------------------------------------------
    # Current multiplication step
    # --------------------------------------------------------

    if current['multiplier_bit'] == 1:
        action_text = f"A is shifted left by {current['shift']} bit position(s) and added to the cumulative sum."
    else:
        action_text = "The current multiplier bit is 0, so the partial product is zero."

    step_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            Current Multiplication Step
        </div>

        <div class="info">

            <span class="metric">Clock / step</span>
            {step} of {bits}<br>

            <span class="metric">Multiplier bit position</span>
            {current['position']}<br>

            <span class="metric">Current multiplier bit</span>
            {current['multiplier_bit']}<br>

            <span class="metric">Left shift</span>
            {current['shift']} bit position(s)

        </div>

        <div class="operation">
            Partial product =
            {current['multiplier_bit']} × A × 2<sup>{current['shift']}</sup>
            = {current['partial_product']}
        </div>

        <div class="small-note">
            {action_text}
        </div>

        <div style="margin-top:8px;">
            <b>Current partial product</b><br>
            <div class="bit-line">
                {render_product_bits(current['partial_binary'], 'bit-product')}
            </div>
        </div>

        <div>
            <b>Cumulative sum</b><br>
            <div class="bit-line">
                {render_product_bits(current['sum_binary'], 'bit-sum')}
            </div>
        </div>

    </div>
    """


    # --------------------------------------------------------
    # Result / history section
    # --------------------------------------------------------

    table_rows = build_step_table(steps[:step], step)

    if step < bits:
        status = f"""
        <div class="small-note">
            The multiplication is still in progress.
            {bits - step} multiplier bit(s) remain to be processed.
        </div>
        """

        final_result = ""

    else:
        status = """
        <div class="small-note">
            <b>The multiplication is complete.</b>
        </div>
        """

        final_result = f"""
        <div class="info" style="margin-top:7px;">
            <b>Final binary product</b><br>
            <div class="bit-line">
                {render_product_bits(final_binary, 'bit-sum')}
            </div>

            <b>Final decimal product</b><br>
            <span class="result-value">
                {A} × {B} = {final_product}
            </span>
        </div>
        """

    result_html.value = f"""
    <div class="section-box">

        <div class="section-title">
            Partial Products and Accumulation
        </div>

        <table class="step-table">

            <tr>
                <th>Step</th>
                <th>Bit position</th>
                <th>B bit</th>
                <th>Shift</th>
                <th>Partial product</th>
                <th>Cumulative sum</th>
            </tr>

            {table_rows}

        </table>

        {status}

        {final_result}

    </div>
    """


# ------------------------------------------------------------
# Reset behavior
# ------------------------------------------------------------

def reset_step_and_update(change):
    if step_slider.value != 1:
        step_slider.value = 1
    else:
        update_display()


def update_word_length(change):
    bits = change['new']

    maximum = 2**bits - 1

    a_slider.max = maximum
    b_slider.max = maximum

    if a_slider.value > maximum:
        a_slider.value = maximum

    if b_slider.value > maximum:
        b_slider.value = maximum

    step_slider.max = bits

    if step_slider.value != 1:
        step_slider.value = 1
    else:
        update_display()


word_length_slider.observe(update_word_length, names='value')
a_slider.observe(reset_step_and_update, names='value')
b_slider.observe(reset_step_and_update, names='value')
step_slider.observe(update_display, names='value')


# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

input_html.layout = Layout(
    width='565px',
    min_width='565px',
    overflow='visible'
)


top_row = HBox(
    [
        input_html,
        controls_box
    ],
    layout=Layout(
        width='900px',
        max_width='900px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='visible'
    )
)


step_html.layout = Layout(
    width='900px',
    max_width='900px',
    overflow='visible'
)


result_html.layout = Layout(
    width='900px',
    max_width='900px',
    overflow='visible'
)


main_layout = VBox(
    [
        top_row,
        step_html,
        result_html
    ],
    layout=Layout(
        width='900px',
        max_width='900px',
        align_items='flex-start',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Initial display
# ------------------------------------------------------------

update_display()

display(style_html)
display(title_html)
display(description_html)
display(main_layout)